In [3]:
import numpy as np
import pickle
from utils_ferm import (
    op_action_tz
)
from utils_CSF_and_UCSF import (
    orthogonal_transform_obt_tbt,
    obt_phys_spatial_to_spin,
    tbt_phys_spatial_to_spin,
    make_short_H_ferm_op
)

from openfermion import (
    FermionOperator,
    jordan_wigner,
    QubitOperator,
    get_sparse_operator,
    normal_ordered
)

from utils_states import (
    somos_to_seniority_config,
    convert_TZ_format_to_sparse_format,
    variance_of_decomp,
    create_composite_state
)
from utils_hamiltonian_qubit import (
    process_hamiltonian_to_remove_irrelevant_terms,
    process_hamiltonian_to_project_onto_symmetric_subspace
)

from utils_partitioning import (
    sorted_insertion_decomposition,
    augment_decomp_with_pauli_x
)

filename = 'data/h2o_data.dump'

with open(filename,'rb') as f:
    (
    list_CSF,
    list_list_ia_CSF,
    list_list_theta_CSF,
    list_sym_CSF_vec,
    list_UCSF_tz,
    tz_states,
    somos_list,
    psi_GS_UCSF_smik,
    list_orb_rot,
    x_orbrot,
    Enuc,
    obt_spatial,
    tbt_spatial
    ) = pickle.load(f)

#Rotate orbitals 
if len(list_orb_rot) != 0:
    obt, tbt = orthogonal_transform_obt_tbt(x_orbrot,list_orb_rot,obt_spatial,tbt_spatial)
else:
    obt = obt_phys_spatial_to_spin(obt_spatial)
    tbt = tbt_phys_spatial_to_spin(tbt_spatial)

Nqubits = obt.shape[0]
Norb    = Nqubits // 2
dim     = 2 ** Nqubits
Nstates = len(tz_states)

print(f"Number of qubits : {Nqubits}")

# generate Hamiltonian

Hferm           = make_short_H_ferm_op(0, obt, tbt)
Hqub            = jordan_wigner(Hferm)
qwc_decomp      = sorted_insertion_decomposition(Hqub, 'qwc')
qwc_decomp_swap = augment_decomp_with_pauli_x(qwc_decomp, Nqubits)

# generate state information

configs      = [somos_to_seniority_config(somos, Norb) for somos in somos_list]
states_full  = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in tz_states]

results = {}

for i in range(Nstates):
    for j in range(Nstates):
        if i >= j:

            bra_tz     = tz_states[i]
            bra        = states_full[i]
            bra_somos  = somos_list[i]
            bra_config = configs[i]

            ket_tz     = tz_states[j]
            ket        = states_full[j]
            ket_somos  = somos_list[j]
            ket_config = configs[j]
            
            Heff_ferm = FermionOperator()
            for term, coef in Hferm.terms.items():
                term_on_ket    = op_action_tz(FermionOperator(term), ket_tz[0], ket_tz[1], ket_tz[2])
                matrix_element = bra.dot(convert_TZ_format_to_sparse_format(dim, term_on_ket).T)
                if np.abs(matrix_element) > 1e-10:
                    Heff_ferm += coef * FermionOperator(term)

            Heff_qub       = jordan_wigner(Heff_ferm)
            qwc_decomp_eff = sorted_insertion_decomposition(Heff_qub, 'qwc')

            if i == j:
                var_decomp_eff = variance_of_decomp(qwc_decomp_eff, ket, Nqubits, general=True)
                var_decomp     = variance_of_decomp(qwc_decomp    , ket, Nqubits, general=False)

            else:
                composite_state     = create_composite_state(bra, ket, Nqubits)
                qwc_decomp_eff_swap = augment_decomp_with_pauli_x(qwc_decomp_eff, Nqubits)
                var_decomp_eff      = variance_of_decomp(qwc_decomp_eff_swap, composite_state, Nqubits + 1, general=True)
                var_decomp          = variance_of_decomp(qwc_decomp_swap    , composite_state, Nqubits + 1, general=False)

            results[(i,j)] = (bra_config, ket_config, var_decomp_eff, var_decomp)
            print("{:<2} {:<4} {:<10} {:<15}".format(i, j, np.real(np.round(var_decomp_eff, 4)), np.real(np.round(var_decomp, 4))))


Number of qubits : 14
0  0    0.415      36.6772        
1  0    0.2345     7845.2453      
1  1    0.2163     35.3914        
2  0    0.3788     7814.3833      
2  1    0.3756     7626.4723      
2  2    0.2893     40.3724        
3  0    0.3992     7925.8378      
3  1    0.2027     7751.9721      
3  2    0.2418     7721.0579      
3  3    0.8394     44.3712        
4  0    0.1318     7976.2859      
4  1    0.7991     7803.2816      
4  2    0.2449     7772.3279      
4  3    0.1772     7895.9665      
4  4    0.4071     39.2445        
5  0    0.2044     7792.819       
5  1    0.4471     7574.7842      
5  2    0.1849     7567.202       
5  3    0.3032     7693.5852      
5  4    0.2813     7747.4548      
5  5    0.0514     33.1888        
6  0    0.2044     7798.8152      
6  1    1.3567     7599.1198      
6  2    0.1875     7573.9864      
6  3    0.2992     7699.3999      
6  4    0.2818     7752.9638      
6  5    6.7752     7540.939       
6  6    0.007      34.5521       

In [24]:
foo = np.array([list(results.values())[i][2] for i in range(len(results.values()))])
num = np.sum(foo ** (1/2)) ** 2
print(num / 1e-3)

foo = np.array([list(results.values())[i][3] for i in range(len(results.values()))])
num = np.sum(foo ** (1/2)) ** 2

print(num / 1e-3)

(1540684.5105248077+0.0010804310725882015j)
(23998363133.45303+0j)
